# graphArt_LM: memory probe -- how long a training example fits on the free T4s?

One training step (forward + backward, exactly as `train_lora.py` does it: fp16, gradient
checkpointing, LoRA, the chunked answer-only loss) at 2k to 20k tokens on one T4, under three
attention set-ups. The first probe ran out of memory at 8k even split over both T4s, which
points at attention running on PyTorch's plain kernel (memory grows with length squared);
this one finds the ceiling of the current set-up and tests the fix. Prints the peak memory and seconds per step of
each, and a verdict: the longest length that stays under 85% of a GPU, and what a full
training run would take at that speed. About 15 minutes, nothing is trained or kept.

**Before running**: *GPU T4 x2*, *Internet on* (to install the libraries and download the
model). No inputs. Then *Save Version -> Save & Run All (Commit)*.

**When it finishes**: the verdict is at the end of the log, after `##### VERDICT`; the numbers
are in `memory_probe.json` in the *Output* tab.

Generated by `scripts/make_kaggle_notebook.py --memory-probe`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the model training will tune.
# LENGTHS      sequence lengths to try, in tokens (prompt + answer). With the code reducer the
#              longest test input is ~12k; timm training examples reach ~12k after reduction.
# MODES        "one": the model on one T4 (as training runs now); "split": its layers spread
#              over both T4s in one process (twice the memory, slower).
# VARIANTS     attention set-ups (the first probe ran out of memory at 8k on both modes):
#              "default" as training runs now; "nogqa" expands Qwen's shared key/value heads
#              first so PyTorch's memory-efficient kernel may run; "efficient" also forbids any
#              other kernel, so it errors out if that kernel cannot run on a T4.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; LENGTHS = [2048, 4096, 6144, 8192, 12288, 16384, 20480]; MODES = ["one"]; VARIANTS = ["default", "nogqa", "efficient"]; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "26ce40dae10dec4e895c079e002489818bbce922c166cc51a5df26f1f44ab146"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. One training step at each length

In [ ]:
cmd = [sys.executable, "scripts/memory_probe.py", "--base", BASE_MODEL, "--lengths", *map(str, LENGTHS),
       "--modes", *MODES, "--variants", *VARIANTS, "--out", str(WORK / "memory_probe.json")]
code = subprocess.run(cmd, env=dict(os.environ, HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")).returncode
print(f"probe exit {code}; done in {(time.time() - START) / 60:.0f} min")